# Tarea 71 - Boston Dataset

El conjunto de datos Boston Housing Dataset es un clásico en el aprendizaje automático (machine learning) utilizado para problemas de regresión. Contiene información recopilada por el Servicio del Censo de EE. UU. sobre viviendas en el área de Boston, Massachusetts. 

Las primeras 13 son características (features) y la última (medv) es el objetivo a predecir: 

Problema de **Regresión**, usaremos **MSELoss** (Error Cuadrático Medio) o L1Loss.

## Imports

In [118]:
import pandas as pd
import matplotlib.pyplot as plt
import os

import torch
from torch.utils.data import Dataset
from torch.utils.data import random_split

import torch.nn.functional as F
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

In [119]:
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')
print('Using device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

torch.backends.cuda.matmul.allow_tf32 = True

# The flag below controls whether to allow TF32 on cuDNN. This flag defaults to True.
torch.backends.cudnn.allow_tf32 = True

Using device: cpu


## Preprocesamiento de datos

In [120]:
boston = pd.read_csv("../data/housing.data", sep='\\s+', header = None)
boston

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273.0,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273.0,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273.0,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273.0,21.0,393.45,6.48,22.0


### Escalar

In [ ]:

class StandardScaler:

    def __init__(self, mean=None, std=None, epsilon=1e-7):
        """Standard Scaler.
        The class can be used to normalize PyTorch Tensors using native functions. The module does not expect the
        tensors to be of any specific shape; as long as the features are the last dimension in the tensor, the module
        will work fine.
        :param mean: The mean of the features. The property will be set after a call to fit.
        :param std: The standard deviation of the features. The property will be set after a call to fit.
        :param epsilon: Used to avoid a Division-By-Zero exception.
        """
        self.mean = mean
        self.std = std
        self.epsilon = epsilon
    def fit(self, values):
        dims = list(range(values.dim() - 1))
        self.mean = torch.mean(values, dim=dims)
        self.std = torch.std(values, dim=dims)
    def transform(self, values):
        return (values - self.mean) / (self.std + self.epsilon)

    def fit_transform(self, values):
        self.fit(values)
        return self.transform(values)

    def __repr__(self):
        return f"mean: {self.mean}, std:{self.std}, epsilon:{self.epsilon}"

### Dataset Boston PyTorch

In [122]:
class BostonDataset(Dataset):
  def __init__(self, src_file, root_dir, transform=None):
    bostonDataset = pd.read_csv(src_file, sep=r'\s+', names=['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV'])
    X = bostonDataset.loc[:, ~bostonDataset.columns.isin(['MEDV'])]
    Y = bostonDataset[["MEDV"]]
    
    # Convertir a tensores
    x_tensor = torch.tensor(X.values, dtype=torch.float32, device=device) # Cuda, añadimos device
    y_tensor = torch.tensor(Y.values, dtype=torch.float32, device=device)
    
    scaler = StandardScaler()
    scaler.fit(x_tensor)
    XScalada = scaler.fit_transform(x_tensor).type(torch.float32)
    self.data = torch.cat((XScalada,y_tensor),1)

    self.data = torch.cat((x_tensor,y_tensor),dim=1)
    self.root_dir = root_dir
    self.transform = transform

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    if torch.is_tensor(idx):
      idx = idx.tolist()
    preds = self.data[idx, :13] 
    spcs = self.data[idx, 13].long()  
    sample = (preds, spcs)
    if self.transform:
      sample = self.transform(sample)
    return sample

### Cargar Boston

In [123]:
dataset = BostonDataset("../data/housing.data", root_dir=None)
display(dataset[0])
print(dataset.data.shape)

(tensor([6.3200e-03, 1.8000e+01, 2.3100e+00, 0.0000e+00, 5.3800e-01, 6.5750e+00,
         6.5200e+01, 4.0900e+00, 1.0000e+00, 2.9600e+02, 1.5300e+01, 3.9690e+02,
         4.9800e+00]),
 tensor(24))

torch.Size([506, 14])


## División en Train y Test

In [124]:
lonxitudeDataset = len(dataset)
tamTrain =int(lonxitudeDataset*0.8)
tamVal = lonxitudeDataset - tamTrain

train_set, val_set = random_split(dataset, [tamTrain, tamVal]) # TODO: aperez: Faltaba esta linea para futuros proyectos

print(f"Tam dataset: {lonxitudeDataset} train: {tamTrain} tamVal: {tamVal}")
train_ldr = torch.utils.data.DataLoader(train_set, batch_size=2,
    shuffle=True, drop_last=False)
# TODO: aperez: num_workers=2 -> S.O: Linux o macOS, PyTorch puede crear "clones" (forks) del proceso principal muy fácilmente para que carguen los datos en paralelo
#               En Windowns maneja el multiprocesamiento de forma diferente (usa algo llamado spawn). Al intentar crear estos subprocesos dentro de un Notebook, 
#               Windows se lía intentando recargar todo tu código desde cero, y los trabajadores colapsan. Ponemos a 0!!

# validation_loader =torch.utils.data.DataLoader(val_set, batch_size=4, shuffle=False, num_workers=2)
validation_loader =torch.utils.data.DataLoader(val_set, batch_size=4, shuffle=False, num_workers=0)   

Tam dataset: 506 train: 404 tamVal: 102


## Creación de Modelo
Ponemos **out_features = 1** por que al ser problema de regresión necesitamos una única salida (precio de cuanto vale la casa)

QUITAMOS el Softmax (en regresión no se usa al final)

In [125]:
class Model(nn.Module):
    def __init__(self, input_dim):
        super(Model, self).__init__()
        self.layer1 = nn.Linear(input_dim, 50)
        self.layer2 = nn.Linear(50, 50)
        self.layer3 = nn.Linear(in_features=50, out_features=1)
        
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        x = self.layer3(x)
        return x

## Instanciar el Modelo
Para problemas de **regresión** como el de Boston, usaremos **MSELoss** (Error Cuadrático Medio) o L1Loss.

En regresión no medimos "aciertos", medimos distancia. Usamos el MSE (Error Cuadrático Medio) para saber qué tan cerca o lejos estamos del precio real

In [126]:
model = Model(13) #TODO: aperez: IMPORTANTE!! Tiene que coincidir con el número de características de entrada (features) que tiene cada patrón
                     # Al ser el datset de Boston , tiene 13 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = torch.nn.MSELoss() 
display(model)

Model(
  (layer1): Linear(in_features=13, out_features=50, bias=True)
  (layer2): Linear(in_features=50, out_features=50, bias=True)
  (layer3): Linear(in_features=50, out_features=1, bias=True)
)

In [127]:
entradaProba,dest = next(iter(train_ldr))
print("Entrada:")
display(entradaProba)
print("Desexada:")
display(dest)
saida = model(entradaProba) # esta é a proba de verdade
print("Saída:")
display(saida)
loss_fn(saida, dest.view(-1, 1).float())

Entrada:


tensor([[1.4051e+01, 0.0000e+00, 1.8100e+01, 0.0000e+00, 5.9700e-01, 6.6570e+00,
         1.0000e+02, 1.5275e+00, 2.4000e+01, 6.6600e+02, 2.0200e+01, 3.5050e+01,
         2.1220e+01],
        [2.3686e+00, 0.0000e+00, 1.9580e+01, 0.0000e+00, 8.7100e-01, 4.9260e+00,
         9.5700e+01, 1.4608e+00, 5.0000e+00, 4.0300e+02, 1.4700e+01, 3.9171e+02,
         2.9530e+01]])

Desexada:


tensor([17, 14])

Saída:


tensor([[17.7534],
        [14.3180]], grad_fn=<AddmmBackward0>)

tensor(0.3344, grad_fn=<MseLossBackward0>)

## Entrenamiento

Para visualizar el entrenamiento necesitamos usar **tensorboard**

```bash
uv add "setuptools==81.0.0"

tensorboard --logdir="notebooks/runs"


In [128]:
def train_one_epoch(epoch_index, tb_writer):
    running_loss = 0.
    last_loss = 0.
    # usamos enumerate para saber en que batch imos
    for i, data in enumerate(train_ldr):
        # Every data instance is an input + label pair
        inputs, labels = data
        # Zero your gradients for every batch!
        optimizer.zero_grad()
        # Make predictions for this batch
        outputs = model(inputs)
        # Compute the loss and its gradients
        loss = loss_fn(outputs, labels.view(-1, 1).float())
        loss.backward()
        # Adjust learning weights
        optimizer.step()
        # Gather data and report
        running_loss += loss.item()
        if i % 10 == 9:
            last_loss = running_loss / 10 # loss per batch
            print('  batch {} loss: {}'.format(i + 1, last_loss))
            running_loss = 0.
    return last_loss

In [ ]:
writer = SummaryWriter('runs/experimento_boston')

EPOCHS = 100
loss_list = torch.zeros((EPOCHS,))
mae_list  = torch.zeros((EPOCHS,))

for epoch in range(EPOCHS):
    print(f'EPOCH {epoch + 1}:')

    model.train(True)
    avg_loss = train_one_epoch(epoch, writer)
    loss_list[epoch] = avg_loss

    # Esto habilitará la pestaña "Histograms" en TensorBoard
    for name, param in model.named_parameters():
        writer.add_histogram(name, param, epoch + 1)

    # Validación
    model.train(False)
    running_vloss = 0.0
    running_mae = 0.0
    
    with torch.no_grad():
        for i, vdata in enumerate(validation_loader):
            vinputs, vlabels = vdata
            voutputs = model(vinputs)
            
            # Ajuste de dimensiones y tipo para regresión
            vlabels_v = vlabels.view(-1, 1).float()
            
            vloss = loss_fn(voutputs, vlabels_v)
            running_vloss += vloss.item()

            # Calculamos el error real (MAE) para TensorBoard
            batch_mae = torch.abs(voutputs - vlabels_v).mean()
            running_mae += batch_mae.item()

    avg_vloss = running_vloss / (i + 1)
    avg_mae = running_mae / (i + 1)
    mae_list[epoch] = avg_mae

    # Enviar datos a Tensorboard
    # Esto creará las gráficas que verás en el navegador
    writer.add_scalars('Loss_Train_vs_Valid', 
                       {'Training': avg_loss, 'Validation': avg_vloss}, 
                       epoch + 1)
    writer.add_scalar('MAE_Validation', avg_mae, epoch + 1)
    
    print(f'LOSS train {avg_loss:.4f} valid {avg_vloss:.4f} | MAE: {avg_mae:.2f}')

# Forzar escritura y cerrar
# Sin esto, TensorBoard puede salir en blanco
writer.flush()
writer.close()
print("Entrenamiento finalizado y datos guardados en TensorBoard.")

EPOCH 1:
  batch 10 loss: 9.925207233428955
  batch 20 loss: 15.196453046798705
  batch 30 loss: 11.73005428314209
  batch 40 loss: 12.778813683986664
  batch 50 loss: 24.68880455605686
  batch 60 loss: 27.382240378856658
  batch 70 loss: 14.827569174766541
  batch 80 loss: 10.76861855685711
  batch 90 loss: 2.3939663499593733
  batch 100 loss: 6.24457488656044
  batch 110 loss: 9.976429997384548
  batch 120 loss: 9.242320638895034
  batch 130 loss: 9.516387820243835
  batch 140 loss: 50.6547097818926
  batch 150 loss: 23.034673404693603
  batch 160 loss: 16.312532003223897
  batch 170 loss: 15.173388680815696
  batch 180 loss: 12.359290385246277
  batch 190 loss: 15.894321596622467
  batch 200 loss: 12.111875700950623
LOSS train 12.1119 valid 19.6343 | MAE: 2.93
EPOCH 2:
  batch 10 loss: 29.184926211833954
  batch 20 loss: 18.337090182304383
  batch 30 loss: 5.071004354953766
  batch 40 loss: 17.919266866706312
  batch 50 loss: 6.175273239612579
  batch 60 loss: 11.832587779313325
  b

## Persistencia del Modelo (Exportación del state_dict)

In [133]:

# Definimos el nombre de la carpeta y creamos la ruta completa
carpeta_modelos = 'modelos'
ruta_archivo = os.path.join(carpeta_modelos, 'modelo_boston_entrenado.pth')

os.makedirs(carpeta_modelos, exist_ok=True)

# Guardamos el "cerebro" de la red en esa nueva ruta
torch.save(model.state_dict(), ruta_archivo)

print(f"¡Modelo guardado exitosamente en la ruta: {ruta_archivo}!")

¡Modelo guardado exitosamente en la ruta: modelos\modelo_boston_entrenado.pth!


## Predicción

In [143]:
# Instanciar y cargar
# IMPORTANTE: input_dim=13 para Boston Housing
model = Model(input_dim=13).to(device)

# Cargamos usando map_location para que no importe si el PC tiene GPU o no
path_modelo = "modelos/modelo_boston_entrenado.pth"
model.load_state_dict(torch.load(path_modelo, map_location=device))
model.eval()

# Ejemplo de predicción con datos reales
# Supongamos que tienes una fila de datos (13 características)
# Un ejemplo de casa "promedio" (valores aproximados)
casa_promedio = torch.tensor([[
    0.006,  # CRIM (Criminalidad baja)
    18.0,   # ZN
    2.31,   # INDUS
    0.0,    # CHAS (No cerca del río)
    0.538,  # NOX
    6.57,   # RM (6.5 habitaciones)
    65.2,   # AGE
    4.09,   # DIS
    1.0,    # RAD
    296.0,  # TAX
    15.3,   # PTRATIO
    396.9,  # B
    4.98    # LSTAT
]]).to(device)

with torch.no_grad():
    resultado = model(casa_promedio)
    print(f"El valor estimado para esta casa es: {resultado.item():.2f} mil euros.")

El valor estimado para esta casa es: 28.50 mil euros.
